# CS 203 — Week 8 Lab: Hyperparameter Tuning
**Software Tools and Techniques for AI | IIT Gandhinagar**

---

In this lab you will implement Grid Search and Random Search **from scratch**, compare them against scikit-learn's built-in versions, and then use **Optuna** (Bayesian Optimisation) on real datasets. You will measure how efficiently each method finds good hyperparameters under a fixed evaluation budget.

### What you'll build
- `MyGridSearchCV` — a full from-scratch grid search with cross-validation
- `MyRandomSearchCV` — a full from-scratch random search with cross-validation
- Side-by-side comparisons with `sklearn.model_selection.GridSearchCV` and `RandomizedSearchCV`
- Optuna studies with visualisation of how the optimiser learns over trials
- A budget-controlled experiment comparing all three strategies

### Datasets used
| Dataset | Task | Why |
|---------|------|-----|
| Iris | Classification | Small, clean, good for sanity-checks |
| Breast Cancer Wisconsin | Classification | Binary, meaningful, fast to train |
| Wine | Classification | Multi-class, compact |
| California Housing | Regression | Larger, real-world numeric features |



## 0  Setup

In [1]:
!pip install optuna --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 7.1 MB/s eta 0:00:00


In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import itertools
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_iris, load_breast_cancer, load_wine, fetch_california_housing
from sklearn.model_selection import (
    KFold, StratifiedKFold, cross_val_score,
    GridSearchCV, RandomizedSearchCV
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from scipy.stats import randint, uniform

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
np.random.seed(SEED)
print("All imports successful ✓")

All imports successful ✓


In [2]:
# Load all datasets once — reuse throughout the lab
iris        = load_iris()
cancer      = load_breast_cancer()
wine        = load_wine()
california  = fetch_california_housing()

X_iris,   y_iris   = iris.data,    iris.target
X_cancer, y_cancer = cancer.data,  cancer.target
X_wine,   y_wine   = wine.data,    wine.target
X_cal,    y_cal    = california.data, california.target

print(f"Iris:            {X_iris.shape}")
print(f"Breast Cancer:   {X_cancer.shape}")
print(f"Wine:            {X_wine.shape}")
print(f"California:      {X_cal.shape}")

NameError: name 'load_iris' is not defined

---
## Part 1 — Grid Search from Scratch

### 1.1  Implement `MyGridSearchCV`

Your class must:
- Accept any scikit-learn compatible estimator, a parameter grid (dict of `{param_name: [values]}`), and `cv` (number of folds).
- Generate **all combinations** of the parameter grid (`itertools.product` will help).
- For each combination, run k-fold cross-validation and record the mean score.
- Expose:
  - `best_params_` — the parameter dict giving the highest CV score
  - `best_score_` — the corresponding score
  - `cv_results_` — a list of dicts, one per combination, with keys `params` and `mean_test_score`

You must use `sklearn.model_selection.KFold` internally (not `cross_val_score` — build the fold loop yourself).

In [3]:
class MyGridSearchCV:
    """
    From-scratch grid search with k-fold cross-validation.

    Parameters
    ----------
    estimator : sklearn-compatible model
    param_grid : dict  {param_name: [value1, value2, ...]}
    cv        : int    number of folds
    scoring   : str    currently only 'accuracy' is required
    """

    def __init__(self, estimator, param_grid, cv=5, scoring='accuracy'):
        self.estimator  = estimator
        self.param_grid = param_grid
        self.cv         = cv
        self.scoring    = scoring

        # Results — populated after fit()
        self.best_params_ = None
        self.best_score_  = -np.inf
        self.cv_results_  = []   # list of {'params': {...}, 'mean_test_score': float}

    def _get_all_combinations(self):
        """
        Return a list of dicts, each representing one parameter combination.
        Example: {'max_depth': 3, 'min_samples_leaf': 1}

        Hint: use itertools.product on the dict values, then zip with the keys.
        """
        # YOUR CODE HERE
        pass

    def _score_single_combination(self, params, X, y):
        """
        Run k-fold CV for one parameter combination.
        Returns the mean accuracy across all folds.

        Steps:
          1. Clone (or re-instantiate) the estimator and set params.
          2. Create a KFold object.
          3. Loop over folds: fit on train split, score on val split.
          4. Return mean of fold scores.

        Hint: use estimator.set_params(**params) to apply hyperparameters,
              and estimator.score(X_val, y_val) to evaluate.
        """
        # YOUR CODE HERE
        pass

    def fit(self, X, y):
        """
        Run the grid search.
        Populates best_params_, best_score_, and cv_results_.
        Returns self.
        """
        # YOUR CODE HERE
        pass

    def get_results_df(self):
        """Return cv_results_ as a sorted DataFrame."""
        df = pd.DataFrame(self.cv_results_)
        return df.sort_values('mean_test_score', ascending=False).reset_index(drop=True)

### 1.2  Test your implementation on the Iris dataset

In [6]:
param_grid_dt = {
    'max_depth':        [1, 2, 3, 5, 7, 10],
    'min_samples_leaf': [1, 2, 4],
}

my_gs = MyGridSearchCV(
    estimator  = DecisionTreeClassifier(random_state=SEED),
    param_grid = param_grid_dt,
    cv         = 5
)
my_gs.fit(X_iris, y_iris)

print(f"Best params : {my_gs.best_params_}")
print(f"Best CV score: {my_gs.best_score_:.4f}")
print()
print(my_gs.get_results_df().head(8))

NameError: name 'X_iris' is not defined

### 1.3  Compare with sklearn's `GridSearchCV`

Run sklearn's `GridSearchCV` on the **same** dataset with the **same** parameter grid and `cv=5`. Then fill in the comparison table below.

In [ ]:
# YOUR CODE HERE
# Run sklearn GridSearchCV with the same estimator, param_grid_dt, and cv=5
# Store the result in a variable called `sk_gs`

sk_gs = ...  # YOUR CODE HERE

print(f"[sklearn] Best params : {sk_gs.best_params_}")
print(f"[sklearn] Best CV score: {sk_gs.best_score_:.4f}")
print()
print(f"[mine]   Best params : {my_gs.best_params_}")
print(f"[mine]   Best CV score: {my_gs.best_score_:.4f}")

In [ ]:
# Visualise: bar chart of mean CV score for every parameter combination
# Use my_gs.get_results_df() and colour bars by max_depth

# YOUR CODE HERE

### 1.4  Grid Search on Breast Cancer — two hyperparameters, an SVM

Grid search an SVM (`SVC`) on the Breast Cancer dataset over the following grid:

```python
param_grid_svm = {
    'C':      [0.01, 0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf'],
}
```

Wrap the SVM in a `Pipeline` with a `StandardScaler` **inside** the pipeline (no leakage!). Report the best params and score. Then plot a **heatmap** of mean CV score (rows = `C`, columns = `kernel`).

In [ ]:
param_grid_svm = {
    'svm__C':      [0.01, 0.1, 1, 10, 100],
    'svm__kernel': ['linear', 'rbf'],
}

pipe_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('svm',    SVC(random_state=SEED))
])

# YOUR CODE HERE — run GridSearchCV with pipe_svm, param_grid_svm, cv=5
# store in gs_svm

gs_svm = ...  # YOUR CODE HERE

print(f"Best params : {gs_svm.best_params_}")
print(f"Best CV score: {gs_svm.best_score_:.4f}")

In [ ]:
# Plot a heatmap: rows = C values, columns = kernel values, colour = mean_test_score
# Hint: pivot gs_svm.cv_results_ into a 2D array

# YOUR CODE HERE

### 1.5  The Curse of Dimensionality — counting evaluations

The grid below is for a Random Forest:

```python
param_grid_rf_large = {
    'n_estimators':    [50, 100, 200, 300, 500],
    'max_depth':       [3, 5, 10, 15, None],
    'min_samples_leaf':[1, 2, 5, 10],
    'max_features':    ['sqrt', 'log2', 0.3, 0.5],
}
```

**Without running any code**, compute:
1. How many unique combinations are there?
2. With `cv=5`, how many total model fits does grid search require?
3. If each fit takes 0.5 seconds, how many minutes would this take?

Then confirm programmatically:

In [ ]:
param_grid_rf_large = {
    'n_estimators':    [50, 100, 200, 300, 500],
    'max_depth':       [3, 5, 10, 15, None],
    'min_samples_leaf':[1, 2, 5, 10],
    'max_features':    ['sqrt', 'log2', 0.3, 0.5],
}

# YOUR CODE HERE
# Compute and print:
# - total unique combinations
# - total model fits with cv=5
# - estimated minutes if each fit takes 0.5 seconds

---
## Part 2 — Random Search from Scratch

### 2.1  Implement `MyRandomSearchCV`

Your class must:
- Accept the same interface as `MyGridSearchCV`, **plus** `n_iter` (number of random samples) and `random_state`.
- The parameter space can contain either a **list** of values (sample uniformly from that list) or a **scipy distribution** with a `.rvs()` method.
- For each of `n_iter` iterations, sample one value per parameter, run k-fold CV, and record the result.
- Expose the same attributes: `best_params_`, `best_score_`, `cv_results_`.

In [ ]:
class MyRandomSearchCV:
    """
    From-scratch random search with k-fold cross-validation.

    Parameters
    ----------
    estimator    : sklearn-compatible model
    param_distributions : dict
        Values are either a list (uniform choice) or a scipy distribution
        (anything with a .rvs(random_state=...) method).
    n_iter       : int   number of random trials
    cv           : int   number of folds
    scoring      : str
    random_state : int
    """

    def __init__(self, estimator, param_distributions, n_iter=20,
                 cv=5, scoring='accuracy', random_state=None):
        self.estimator           = estimator
        self.param_distributions = param_distributions
        self.n_iter              = n_iter
        self.cv                  = cv
        self.scoring             = scoring
        self.random_state        = random_state

        self.best_params_ = None
        self.best_score_  = -np.inf
        self.cv_results_  = []

    def _sample_params(self, rng):
        """
        Sample one value per hyperparameter.
        - If the value is a list: choose uniformly at random.
        - If the value has a .rvs() method (scipy distribution): call .rvs(random_state=rng).
        Returns a dict {param_name: sampled_value}.
        """
        # YOUR CODE HERE
        pass

    def _cross_val_score(self, params, X, y):
        """
        K-fold CV for one sampled combination.
        Same logic as MyGridSearchCV._score_single_combination.
        """
        # YOUR CODE HERE
        pass

    def fit(self, X, y):
        """
        Run n_iter random trials.
        Populates best_params_, best_score_, cv_results_.
        Returns self.
        """
        # YOUR CODE HERE
        pass

    def get_results_df(self):
        df = pd.DataFrame(self.cv_results_)
        return df.sort_values('mean_test_score', ascending=False).reset_index(drop=True)

### 2.2  Test on Iris

In [ ]:
param_dist_dt = {
    'max_depth':        [1, 2, 3, 5, 7, 10, 15, 20],
    'min_samples_leaf': randint(1, 10),     # scipy distribution
    'min_samples_split': randint(2, 20),
}

my_rs = MyRandomSearchCV(
    estimator            = DecisionTreeClassifier(random_state=SEED),
    param_distributions  = param_dist_dt,
    n_iter               = 30,
    cv                   = 5,
    random_state         = SEED
)
my_rs.fit(X_iris, y_iris)

print(f"Best params : {my_rs.best_params_}")
print(f"Best CV score: {my_rs.best_score_:.4f}")
print()
print(my_rs.get_results_df().head(8))

### 2.3  Compare with sklearn's `RandomizedSearchCV`

Run sklearn's version with the **same** parameter distributions, `n_iter=30`, `cv=5`, and `random_state=SEED`. Print both best scores and params side by side.

In [ ]:
# YOUR CODE HERE — store in sk_rs
sk_rs = ...  # YOUR CODE HERE

print(f"[sklearn] Best params : {sk_rs.best_params_}")
print(f"[sklearn] Best CV score: {sk_rs.best_score_:.4f}")
print()
print(f"[mine]   Best params : {my_rs.best_params_}")
print(f"[mine]   Best CV score: {my_rs.best_score_:.4f}")

### 2.4  Convergence plot — best score vs number of trials

For your `MyRandomSearchCV`, plot how the **best score seen so far** evolves as you go from trial 1 to trial 30. Do the same for your `MyGridSearchCV` results (pretend they arrive in a fixed order).

Hint: compute `np.maximum.accumulate` on the list of scores in the order they were evaluated.

In [ ]:
# YOUR CODE HERE
# Extract scores in evaluation order from my_rs.cv_results_ and my_gs.cv_results_
# Plot cumulative best for both on the same axes

---
## Part 3 — Grid vs Random: The Bergstra & Bengio Experiment

The lecture showed that random search covers the important dimension more densely than grid search when one hyperparameter matters much more than the other. Here you will **reproduce that intuition empirically**.

### 3.1  Fixed budget comparison on Wine dataset

Budget = **25 evaluations** with `cv=5`. Use a `RandomForestClassifier`.

- Grid search: design a grid with exactly 25 combinations (e.g., 5 × 5 or 5 × 5 × 1).
- Random search: 25 random samples from a **wide** continuous distribution.

Report: best score, best params, and wall-clock time for each.

In [ ]:
BUDGET = 25

# --- Grid search (25 combinations) ---
# Design your grid here — it must have exactly BUDGET combinations
param_grid_wine = {
    # YOUR CODE HERE
}
# Verify combination count
n_combos = 1
for v in param_grid_wine.values():
    n_combos *= len(v)
assert n_combos == BUDGET, f"Expected {BUDGET} combos, got {n_combos}"

t0 = time.time()
gs_wine = GridSearchCV(
    RandomForestClassifier(random_state=SEED),
    param_grid_wine, cv=5, n_jobs=-1
)
gs_wine.fit(X_wine, y_wine)
t_grid = time.time() - t0

print(f"[Grid]   best={gs_wine.best_score_:.4f}  params={gs_wine.best_params_}  time={t_grid:.2f}s")

In [ ]:
# --- Random search (25 iterations, wide distributions) ---
param_dist_wine = {
    # YOUR CODE HERE — use scipy distributions for at least two params
}

t0 = time.time()
rs_wine = RandomizedSearchCV(
    RandomForestClassifier(random_state=SEED),
    param_dist_wine, n_iter=BUDGET, cv=5, n_jobs=-1, random_state=SEED
)
rs_wine.fit(X_wine, y_wine)
t_rand = time.time() - t0

print(f"[Random] best={rs_wine.best_score_:.4f}  params={rs_wine.best_params_}  time={t_rand:.2f}s")

### 3.2  Coverage plot

Pick two hyperparameters (e.g., `n_estimators` and `max_depth`). Scatter-plot all 25 evaluated points for grid search and random search on the same axes (different colours). What do you observe?

In [ ]:
# YOUR CODE HERE
# Extract (n_estimators, max_depth) from gs_wine.cv_results_['params']
#                                 and rs_wine.cv_results_['params']
# Scatter both sets on the same axes, coloured differently

### 3.3  Repeated trials — mean and standard deviation of best score

Random search has variance across different random seeds. Repeat it 10 times (with seeds 0–9) and report the mean ± std of `best_score_`. Do the same for your `MyRandomSearchCV`.

In [ ]:
# YOUR CODE HERE
sklearn_scores = []
my_scores      = []

for seed in range(10):
    # sklearn RandomizedSearchCV with this seed
    # YOUR CODE HERE
    sklearn_scores.append(...)  # best_score_

    # MyRandomSearchCV with this seed
    # YOUR CODE HERE
    my_scores.append(...)  # best_score_

print(f"[sklearn] mean={np.mean(sklearn_scores):.4f}  std={np.std(sklearn_scores):.4f}")
print(f"[mine]   mean={np.mean(my_scores):.4f}  std={np.std(my_scores):.4f}")

---
## Part 4 — Bayesian Optimisation with Optuna

### 4.1  Your first Optuna study

Write an Optuna objective function for a `RandomForestClassifier` on the Breast Cancer dataset. Tune:
- `n_estimators`: integer between 50 and 500
- `max_depth`: integer between 2 and 30
- `min_samples_leaf`: integer between 1 and 20
- `max_features`: float between 0.1 and 1.0

Run for 60 trials. Print the best value and best params.

In [ ]:
def objective_cancer(trial):
    """
    Optuna objective: return the 5-fold CV accuracy on Breast Cancer.
    """
    # YOUR CODE HERE
    # Suggest hyperparameters using trial.suggest_int / trial.suggest_float
    # Build the model, cross-validate, return mean score
    pass


study_cancer = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study_cancer.optimize(objective_cancer, n_trials=60)

print(f"Best CV accuracy : {study_cancer.best_value:.4f}")
print(f"Best params      : {study_cancer.best_params}")

### 4.2  Visualise how Optuna learns over trials

Plot the **objective value per trial** and the **cumulative best** on the same figure. The cumulative best should be monotonically non-decreasing.

In [ ]:
# YOUR CODE HERE
# Extract per-trial values from study_cancer.trials
# Each trial has .value and .number attributes

### 4.3  Parameter importance

Optuna can compute how much each hyperparameter contributed to the objective variance. Use `optuna.importance.get_param_importances` and plot a horizontal bar chart.

In [ ]:
# YOUR CODE HERE
importances = optuna.importance.get_param_importances(study_cancer)
# Plot a horizontal bar chart with param names on y-axis and importance on x-axis

### 4.4  Optuna on a regression task — California Housing

Switch to regression. Tune a `GradientBoostingRegressor` on the California Housing dataset. Maximise the negative RMSE (or equivalently minimise RMSE — set `direction='minimize'` and return the RMSE).

Hyperparameters to tune:
- `n_estimators`: 50 – 400
- `learning_rate`: log-uniform between 1e-3 and 1.0 (use `trial.suggest_float(..., log=True)`)
- `max_depth`: 2 – 10
- `subsample`: 0.5 – 1.0

Run 50 trials.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_val_score as cvs

def objective_cal(trial):
    """
    Return RMSE (to be minimised).
    Use neg_root_mean_squared_error scoring with cross_val_score, then negate.
    """
    # YOUR CODE HERE
    pass


study_cal = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED))
study_cal.optimize(objective_cal, n_trials=50)

print(f"Best RMSE   : {study_cal.best_value:.4f}")
print(f"Best params : {study_cal.best_params}")

---
## Part 5 — Head-to-Head: Grid vs Random vs BayesOpt

### 5.1  Benchmark under a shared evaluation budget

Use the **Wine** dataset with a `RandomForestClassifier`. Fix the budget to **N = 40 evaluations** (each evaluation = one (params, cv) pair → 5 model fits). Compare:

| Method | How to fix budget |
|--------|------------------|
| Grid search | Design a grid with ≤ 40 combinations |
| Random search | `n_iter=40` |
| Optuna (BayesOpt) | `n_trials=40` |

Record: best CV score and wall-clock time for each. Report in a summary table.

In [ ]:
BUDGET = 40
results_summary = {}  # will be filled in

# --- 1. Grid Search ---
# YOUR CODE HERE
results_summary['GridSearch'] = {'best_score': ..., 'time_s': ...}

# --- 2. Random Search ---
# YOUR CODE HERE
results_summary['RandomSearch'] = {'best_score': ..., 'time_s': ...}

# --- 3. Optuna (BayesOpt) ---
# YOUR CODE HERE
results_summary['BayesOpt'] = {'best_score': ..., 'time_s': ...}

# Print a formatted summary table
print(f"{'Method':<15} {'Best CV Score':>13} {'Time (s)':>10}")
print('-' * 42)
for method, vals in results_summary.items():
    print(f"{method:<15} {vals['best_score']:>13.4f} {vals['time_s']:>10.2f}")

### 5.2  Learning curves — best score vs number of evaluations

For the same experiment above, plot **cumulative best score** as a function of trial number (1 to 40) for all three methods on the same axes. This is the canonical way to compare search strategies.

Hint: for Optuna, use `[t.value for t in study.trials]` to get the per-trial scores.

In [ ]:
# YOUR CODE HERE
# Plot cumulative best for all three methods
# Label axes clearly: x = "Number of evaluations", y = "Best CV score so far"

### 5.3  Multi-dataset benchmark

Repeat the 3-way comparison (budget = 30) for **all four datasets** (Iris, Breast Cancer, Wine, and for California Housing use negative RMSE as the score). Produce a summary DataFrame with rows = datasets and columns = methods.

In [ ]:
BUDGET = 30
datasets = {
    'Iris':          (X_iris,   y_iris,   'classification'),
    'BreastCancer':  (X_cancer, y_cancer, 'classification'),
    'Wine':          (X_wine,   y_wine,   'classification'),
    'California':    (X_cal,    y_cal,    'regression'),
}

# YOUR CODE HERE
# For each dataset, run Grid / Random / Optuna and store best scores
# Display a DataFrame at the end

---
## Part 6 — Advanced Optuna: Pruning and Categorical Hyperparameters

### 6.1  Categorical hyperparameters — choosing the model family

Use `trial.suggest_categorical` to let Optuna choose **both** the model family **and** its hyperparameters in a single study. Include at least: `DecisionTree`, `RandomForest`, `GradientBoosting`, `KNN`. Run on Breast Cancer for 60 trials.

In [ ]:
def objective_automl(trial):
    """
    Let Optuna choose the model family and tune it.
    Return 5-fold CV accuracy on Breast Cancer.

    Structure:
      classifier = trial.suggest_categorical('classifier', ['DT', 'RF', 'GB', 'KNN'])
      if classifier == 'DT':
          ... suggest DT params ...
      elif classifier == 'RF':
          ... suggest RF params ...
      ...
    """
    # YOUR CODE HERE
    pass


study_automl = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study_automl.optimize(objective_automl, n_trials=60)

print(f"Best CV accuracy : {study_automl.best_value:.4f}")
print(f"Best params      : {study_automl.best_params}")

In [ ]:
# Count how many trials each classifier family was selected
# and plot a bar chart

# YOUR CODE HERE

### 6.2  Reproducing a study — `random_state` and Optuna seeds

Re-run your `objective_cancer` study with `n_trials=60` **twice**, both times using the same `TPESampler(seed=SEED)`. Confirm that both runs produce **exactly the same** sequence of trial values. What does this tell you about reproducibility?

In [ ]:
# YOUR CODE HERE
# Run two studies, extract [t.value for t in study.trials] from each
# Assert they are equal

### 6.3  Multi-seed variance report

Run Optuna on the Wine dataset (30 trials each) using 10 different seeds (0–9). Report the mean ± std of the best CV score across seeds. Do the same for Random Search with the same budget.

In [ ]:
# YOUR CODE HERE
optuna_scores = []
random_scores = []

for seed in range(10):
    # Optuna run
    # YOUR CODE HERE
    optuna_scores.append(...)  # best_value

    # RandomizedSearchCV run
    # YOUR CODE HERE
    random_scores.append(...)  # best_score_

print(f"[Optuna] mean={np.mean(optuna_scores):.4f}  std={np.std(optuna_scores):.4f}")
print(f"[Random] mean={np.mean(random_scores):.4f}  std={np.std(random_scores):.4f}")

---
## Part 7 — Pipelines, Preprocessing, and Data Leakage

### 7.1  The wrong way vs the right way

Demonstrate data leakage: scale the entire Breast Cancer dataset **before** running cross-validation, and compare the CV score to scaling **inside** a Pipeline. The scores should differ — explain why.

In [ ]:
from sklearn.svm import SVC

# --- WRONG: scale before CV (leaky) ---
scaler = StandardScaler()
X_cancer_scaled = scaler.fit_transform(X_cancer)  # sees ALL data including future val folds!

leaky_score = cross_val_score(
    SVC(C=1, kernel='rbf', random_state=SEED),
    X_cancer_scaled, y_cancer, cv=5
).mean()

# --- RIGHT: scale inside Pipeline ---
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svm',    SVC(C=1, kernel='rbf', random_state=SEED))
])
correct_score = cross_val_score(pipe, X_cancer, y_cancer, cv=5).mean()

print(f"Leaky  (wrong) CV accuracy: {leaky_score:.4f}")
print(f"Correct (Pipeline) accuracy: {correct_score:.4f}")
print(f"Difference: {leaky_score - correct_score:+.4f}")

### 7.2  Tuning a Pipeline with `GridSearchCV`

Build a `Pipeline(StandardScaler → SVC)` and tune it with `GridSearchCV` on Breast Cancer. Use the `svm__` prefix for the SVM parameters.

```python
param_grid_pipe = {
    'svm__C':      [0.01, 0.1, 1, 10, 100],
    'svm__kernel': ['linear', 'rbf'],
    'svm__gamma':  ['scale', 'auto'],
}
```

In [ ]:
param_grid_pipe = {
    'svm__C':      [0.01, 0.1, 1, 10, 100],
    'svm__kernel': ['linear', 'rbf'],
    'svm__gamma':  ['scale', 'auto'],
}

pipe2 = Pipeline([
    ('scaler', StandardScaler()),
    ('svm',    SVC(random_state=SEED))
])

# YOUR CODE HERE — GridSearchCV on pipe2 with param_grid_pipe
gs_pipe = ...  # YOUR CODE HERE

print(f"Best params : {gs_pipe.best_params_}")
print(f"Best CV score: {gs_pipe.best_score_:.4f}")

---
## Part 8 — Capstone: Full Tuning Workflow

Implement the complete tuning workflow from the lecture on the **Wine** dataset:

1. **Dummy baseline** — what does a random classifier score?
2. **Simple model** — Logistic Regression with default params.
3. **Tuned strong model** — Random Forest, tuned with Optuna (50 trials).
4. **DIY AutoML** — loop over at least 4 model families, each with its own grid, pick the winner.

Print a clearly formatted leaderboard at the end.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import ExtraTreesClassifier

leaderboard = {}  # { 'method_name': cv_score }

# 1. Dummy baseline
# YOUR CODE HERE
leaderboard['DummyClassifier'] = ...

# 2. Logistic Regression (default)
# YOUR CODE HERE
leaderboard['LogisticRegression (default)'] = ...

# 3. Random Forest tuned with Optuna (50 trials)
# YOUR CODE HERE
leaderboard['RandomForest (Optuna 50 trials)'] = ...

# 4. DIY AutoML — at least 4 model families
model_configs = {
    # YOUR CODE HERE
    # 'ModelName': (ModelClass(), param_grid_dict)
}
for name, (model, params) in model_configs.items():
    gs = GridSearchCV(model, params, cv=5, n_jobs=-1)
    gs.fit(X_wine, y_wine)
    leaderboard[f'AutoML-{name}'] = gs.best_score_

# Print leaderboard sorted by score
print(f"\n{'Method':<40} {'CV Score':>10}")
print('─' * 52)
for method, score in sorted(leaderboard.items(), key=lambda x: x[1], reverse=True):
    print(f"{method:<40} {score:>10.4f}")

### 8.1  Reflection questions

Answer the following in the markdown cell below (no code needed):

1. Your `MyGridSearchCV` and sklearn's `GridSearchCV` should give the same best score. If they differ slightly, what could cause the discrepancy?
2. In the convergence plot (Part 5.2), which method found a good score with the fewest evaluations? What does this tell you?
3. A colleague suggests running Optuna with 500 trials on every project. Give two reasons why this might be wasteful.
4. Why does putting `StandardScaler` inside a `Pipeline` prevent data leakage during cross-validation?
5. Looking at the parameter importance chart (Part 4.3), which hyperparameter mattered most? Does this match your intuition?

*(Write your answers here)*

---
## Bonus — Optuna Visualisation Dashboard

Optuna ships built-in interactive plots. Run the cells below on any completed study to explore the results visually (works best in a Jupyter/Colab environment with `plotly` installed).

In [ ]:
!pip install plotly --quiet

In [ ]:
from optuna.visualization import (
    plot_optimization_history,
    plot_param_importances,
    plot_parallel_coordinate,
    plot_contour,
)

# Change `study_cancer` to any study you've created above
plot_optimization_history(study_cancer).show()
plot_param_importances(study_cancer).show()
plot_parallel_coordinate(study_cancer).show()

In [ ]:
# Contour plot for two parameters of your choice
plot_contour(study_cancer, params=['n_estimators', 'max_depth']).show()